In [1]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, 'patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio_from_raw70m.csv'), index_col=0)
        # cell_label = cell_label.apply(lambda x: x*100.0)
        
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        label_index_set = set(label_df.index)
        patch_index_set = set(patch_list)
        and_set = label_index_set & patch_index_set

        patch_list = list(and_set)
        self.label_df = label_df.loc[patch_list]
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, 'patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        label = self.label_df.loc[patch_id].values
        label = torch.Tensor(label)

        return patch_id, data, label

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [2]:
# prepare necessary arguments and WSI sample list

tif_list = '/data1/r20user3/shared_project/Hist2Cell/data/her2st'
tif_list = glob(tif_list + '/*[!xlsx|ipynb]')
tif_list.sort()
tif_list

['/data1/r20user3/shared_project/Hist2Cell/data/her2st/A1',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/A2',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/A3',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/A4',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/A5',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/A6',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B1',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B2',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B3',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B4',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B5',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/B6',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C3',
 '/data1/r20user3/shared_project/Hist2Cell/data/her2st/C4',
 '/data1/r20user3/shared_project/Hist2Ce

In [3]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['A1',
 'A2',
 'A3',
 'A4',
 'A5',
 'A6',
 'B1',
 'B2',
 'B3',
 'B4',
 'B5',
 'B6',
 'C1',
 'C2',
 'C3',
 'C4',
 'C5',
 'C6',
 'D1',
 'D2',
 'D3',
 'D4',
 'D5',
 'D6',
 'E1',
 'E2',
 'E3',
 'F1',
 'F2',
 'F3',
 'G1',
 'G2',
 'G3',
 'H1',
 'H2',
 'H3']

In [4]:
# prepare my GPU

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [6]:
batch_size = 512
patch_dir = "/data1/r20user3/shared_project/Hist2Cell/data/her2st"

graphs = dict()
for slide in test_slides:
    print(slide)
    test_data = PTC_cell(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)

    spots_coord = pd.read_csv(os.path.join("/data1/r20user3/shared_project/Hist2Cell/data/her2st", slide, "spots.csv"), index_col=0)

    with torch.no_grad():
        # model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []

        for name, data, label in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            test_label_array.append(label.cpu().detach().numpy())
            
            # features, output = model(data)
            # test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            test_prediction_array.append(data.cpu().detach().numpy())

    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    for i in range(len(test_label_array)):
        if len(test_label_array[i].shape) <= 1:
            test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_prediction_array = np.concatenate(test_prediction_array)
    test_label_array = np.concatenate(test_label_array)
    
    test_names = list()
    for names in test_name_array:
        test_names = test_names+names
    test_node_x_y = list()
    for item in test_names:
        test_node_x_y.append([int(item.split('x')[0]), int(item.split('x')[1])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 2 or x2 >= x1 + 2 or y2 <= y1 - 2 or y2 >= y1 + 2:
                    continue
                else:
                    adj[i][j] = 1.0

    graphs[slide] = {
        'features': test_prediction_array,
        'labels': test_label_array,
        'adj': adj,
        'names': test_names,
        'coord': spots_coord.loc[test_names].values,
    }

    print("OK")

A1
OK
A2
OK
A3
OK
A4
OK
A5
OK
A6
OK
B1
OK
B2
OK
B3
OK
B4
OK
B5
OK
B6
OK
C1
OK
C2
OK
C3
OK
C4
OK
C5
OK
C6
OK
D1
OK
D2
OK
D3
OK
D4
OK
D5
OK
D6
OK
E1
OK
E2
OK
E3
OK
F1
OK
F2
OK
F3
OK
G1
OK
G2
OK
G3
OK
H1
OK
H2
OK
H3
OK


In [7]:
graphs['A1']['features'].shape

(346, 3, 224, 224)

In [8]:
graphs['A1']['coord'].shape

(346, 2)

In [9]:
graphs['A1']['names']

['17x25',
 '24x27',
 '21x16',
 '17x27',
 '23x26',
 '16x18',
 '5x15',
 '24x23',
 '4x24',
 '24x22',
 '19x17',
 '7x14',
 '14x18',
 '21x14',
 '16x16',
 '4x16',
 '24x21',
 '24x25',
 '11x17',
 '22x24',
 '23x23',
 '6x18',
 '21x12',
 '4x18',
 '21x20',
 '11x23',
 '12x24',
 '9x26',
 '5x18',
 '20x13',
 '12x27',
 '7x17',
 '23x24',
 '7x20',
 '18x15',
 '6x21',
 '9x25',
 '14x13',
 '12x15',
 '13x26',
 '12x18',
 '14x25',
 '9x23',
 '12x21',
 '22x23',
 '18x16',
 '6x15',
 '17x26',
 '11x22',
 '25x16',
 '12x17',
 '19x22',
 '6x25',
 '23x20',
 '25x13',
 '25x21',
 '21x19',
 '22x27',
 '9x22',
 '8x27',
 '7x26',
 '12x13',
 '9x15',
 '12x12',
 '15x27',
 '15x13',
 '22x22',
 '9x19',
 '15x20',
 '17x22',
 '14x21',
 '25x22',
 '5x16',
 '7x15',
 '26x14',
 '24x16',
 '7x19',
 '13x12',
 '13x28',
 '12x26',
 '24x18',
 '20x26',
 '17x17',
 '15x24',
 '15x22',
 '5x27',
 '14x16',
 '9x14',
 '15x23',
 '13x17',
 '16x23',
 '16x15',
 '13x21',
 '26x16',
 '15x14',
 '22x20',
 '16x27',
 '14x14',
 '23x19',
 '22x18',
 '10x23',
 '8x19',
 '11x1

In [10]:
graph = graphs
for slide in graph:
    print(slide)

A1
A2
A3
A4
A5
A6
B1
B2
B3
B4
B5
B6
C1
C2
C3
C4
C5
C6
D1
D2
D3
D4
D5
D6
E1
E2
E3
F1
F2
F3
G1
G2
G3
H1
H2
H3


In [11]:
from torch import Tensor
import torch
import os

for slide_name in graph:
    x = Tensor(graph[slide_name]['features'])
    from torch_geometric.utils import dense_to_sparse
    adj = Tensor(graph[slide_name]['adj'])
    edge_index, _ = dense_to_sparse(adj)
    y = Tensor(graph[slide_name]['labels'])
    from torch_geometric.data import Data
    pos = Tensor(graph[slide_name]['coord'])

    data = Data(x=x, edge_index=edge_index, y=y, pos=pos)
    
    torch.save(data, os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/her2st_from_raw70m", slide_name+".pt"))

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import torch

data1 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/her2st_from_raw70m/A1.pt")
data2 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/her2st_from_raw70m/A2.pt")

In [13]:
from torch_geometric.data import Batch

data = Batch.from_data_list([data1, data2])
data

DataBatch(x=[671, 3, 224, 224], edge_index=[2, 5493], y=[671, 289], pos=[671, 2], batch=[671], ptr=[3])

In [14]:
from random import shuffle
from torch_geometric.loader import NeighborLoader
import torch_geometric

torch_geometric.typing.WITH_PYG_LIB = False

from torch_geometric.seed import seed_everything
seed_everything(0)

loader = NeighborLoader(
    data,
    # Sample 30 neighbors for each node for 2 iterations
    num_neighbors=[-1, -1],
    # Use a batch size of 128 for sampling training nodes
    batch_size=1,
    directed=False,
    input_nodes=None,
    # disjoint=True,
    shuffle=True
)

i = 0
for sampled_data in loader:
    print(sampled_data.input_id)
    i = i + 1
    if i > 10:
        break

tensor([483])
tensor([134])
tensor([431])
tensor([133])
tensor([492])
tensor([200])
tensor([463])
tensor([372])
tensor([28])
tensor([245])
tensor([17])


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "


In [15]:
sampled_data

DataBatch(x=[18, 3, 224, 224], edge_index=[2, 112], y=[18, 289], pos=[18, 2], ptr=[3], n_id=[18], e_id=[112], input_id=[1], batch_size=1)

In [16]:
sampled_data.pos

tensor([[5037.7402, 5249.1099],
        [4816.7300, 5472.1602],
        [5042.7402, 5025.8398],
        [5228.5400, 5464.9399],
        [4817.6001, 5243.6401],
        [4809.3398, 5024.3101],
        [5224.1899, 5249.1099],
        [5225.7100, 5030.2100],
        [5024.2700, 5462.1001],
        [4564.4302, 5453.1401],
        [5044.4800, 5684.4902],
        [4555.0898, 5672.4702],
        [4549.6499, 5239.4902],
        [5242.8799, 4801.7002],
        [4804.3398, 4811.7598],
        [5023.8301, 4802.3501],
        [4547.2598, 5017.9702],
        [4558.5601, 4793.8301]])

: 